# Klimabegriffe auf deutschsprachigen Titelseiten

## Ziel
Dieses Notebook ist die Hauptanalyse der Studienarbeit. Es untersucht die Entwicklung von `Klimawandel`, `Klimakrise` und `Klimaschutz` auf deutschsprachigen Online-Titelseiten im konsistenten Analysezeitraum.

## Leitfragen
- Wie verteilen sich die drei Begriffe insgesamt?
- Wie verändern sich ihre relativen Anteile über Quartale und Monate?
- Verschiebt sich die Sprache zwischen neutralen und alarmistischen Begriffen?


In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from itertools import combinations

# Sphinx-Dokumentationskontext

notebook_dir = (
    os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
 )
sys.path.append(os.path.join(notebook_dir, "..", "pylib"))

from analysis_utils import map_term_category
from config import (
    DEFAULT_END_DATE,
    DWH_DB_PATH,
    PLOTS_DIR,
    SCRAPER_CUTOFF,
 )
from handle_sqlite import read_table_as_dataframe
from plotting_utils import (
    apply_plot_style,
    counts_for_period,
    delta_between_periods,
    scatter_delta,
    shares_from_counts,
 )

apply_plot_style()


In [ ]:
# Ausgabepfad
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
plot_dir = PLOTS_DIR


In [ ]:
# Aufbereitete Tabellen aus SQLite laden
df_context = read_table_as_dataframe("context_processed", str(DWH_DB_PATH))
df_meta = read_table_as_dataframe("newspapers", str(DWH_DB_PATH))

df_meta["data_published"] = pd.to_datetime(df_meta["data_published"])

print(f"Context rows: {len(df_context)}")
print(f"Metadata rows: {len(df_meta)}")


In [ ]:
analysis_start = SCRAPER_CUTOFF
analysis_end = DEFAULT_END_DATE

df_meta_filtered = df_meta[df_meta["data_published"] >= analysis_start].copy()
if analysis_end is not None:
    df_meta_filtered = df_meta_filtered[df_meta_filtered["data_published"] <= analysis_end].copy()

analysis_end_label = analysis_end.date() if analysis_end is not None else "open end"
print(f"Total rows: {len(df_meta)}")
print(f"Rows from {analysis_start.date()} to {analysis_end_label}: {len(df_meta_filtered)}")


In [ ]:
# Analysefenster anzeigen
daily = df_meta.groupby("data_published").size()

fig, ax = plt.subplots(figsize=(14, 4))

# Analysefenster kontrastreich markieren
outside_color = "#E8E8E8"
inside_color = "#111111"
colors = [
    inside_color if analysis_start <= pd.to_datetime(d) <= analysis_end else outside_color
    for d in daily.index
  ]

ax.bar(daily.index, daily.values, color=colors, edgecolor="white", linewidth=0.35, width=1.0)
ax.axvspan(analysis_start, analysis_end, color="#000000", alpha=0.06, label="Analysis window")

ax.axvline(
    analysis_start,
    color="black",
    linestyle="--",
    linewidth=1.2,
    label=f"Cutoff {analysis_start.date()}",
)
ax.axvline(
    analysis_end,
    color="black",
    linestyle=":",
    linewidth=1.2,
    label=f"End {analysis_end.date()}",
)

ax.set_title("Daily scraping volume with analysis window highlighted")
ax.set_xlabel("Date")
ax.set_ylabel("Newspaper-days")
ax.legend(loc="upper right", frameon=True)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()


In [ ]:
# Merge: Metadaten bleiben vollständig, Kontext wird ergänzt, wo vorhanden
df = df_meta_filtered[['newspaper_id', 'data_published']].merge(df_context, on='newspaper_id', how='left')

print(f"Zeitraum: {df['data_published'].min()} bis {df['data_published'].max()}")
print(f"Einträge insgesamt: {len(df_meta)} Nennungen")
print(f"Daten von {analysis_start.date()} bis {analysis_end_label}: {len(df)} Nennungen")


In [ ]:
# Finale Mapping-Definition für die Hauptanalyse, nach dem processing:
# Klimawandel: wandel, Klimakrise: krise, Klimaschutz: schutz
SUFFIX_TO_BEGRIFF = {
    'wandel': 'Klimawandel',
    'krise':  'Klimakrise',
    'schutz': 'Klimaschutz',
}

df['begriff'] = (
    df['suffix_lemma']
    .str.lower()
    .str.strip()
    .map(SUFFIX_TO_BEGRIFF)
)

print("\nVerwendete finale Suffixe:")
for suffix, begriff in SUFFIX_TO_BEGRIFF.items():
    print(f"- {begriff}: {suffix}")

df_drei = df[df['begriff'].notna()].copy()
print(f"Nennungen der drei Begriffe (finale Auswahl): {len(df_drei)}")
print(df_drei['begriff'].value_counts())


## 1. Gesamtverteilung der Begriffe


In [ ]:
# Absolute Häufigkeiten
counts = df_drei['begriff'].value_counts()
total = counts.sum()
percentages = (counts / total * 100).round(2)

fig, ax = plt.subplots(figsize=(7, 4))

# Zurückhaltende Graustufen für die drei Begriffe
bar_colors = ['#1A1A1A', '#7A7A7A', '#C9C9C9']
bars = ax.bar(counts.index, counts.values, color=bar_colors, edgecolor='white', linewidth=0.6)

# Prozentzahl über jeden Balken
for bar, pct in zip(bars, percentages.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(counts.max() * 0.02, 3),
        f'{pct}%',
        ha='center',
        va='bottom',
        fontsize=10,
        color='black'
    )

ax.set_title('Gesamtverteilung der Begriffe (21.04.2022 – 08.02.2025)')
ax.set_ylabel('Absolute Häufigkeit')
ax.set_xlabel('Begriff')
ax.set_ylim(0, counts.max() * 1.18)  # Platz für Labels
ax.grid(axis='y', alpha=0.2)

for axis in (ax,):
    axis.spines['top'].set_visible(False)
    axis.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()


## 2. Zeitliche Entwicklung nach Quartalen


In [ ]:
# Quartal berechnen
df_drei['quarter'] = df_drei['data_published'].dt.to_period('Q')
# Gruppieren: Quartal x Begriff
quarterly_counts = df_drei.groupby(['quarter', 'begriff']).size().unstack(fill_value=0)
# Relative Anteile pro Quartal berechnen
quarterly_total = quarterly_counts.sum(axis=1)
quarterly_pct = quarterly_counts.div(quarterly_total, axis=0) * 100
print("\n=== ANALYSE 2: Relative Anteile pro Quartal (%) ===")
print(quarterly_pct.round(2).to_string())


In [ ]:
# Auch absolute Zahlen pro Quartal zeigen
print("\n=== ANALYSE 2b: Absolute Häufigkeiten pro Quartal ===")
quarterly_counts['Gesamt'] = quarterly_total
print(quarterly_counts.to_string())


## 3. Neutral vs. alarmistisch


In [ ]:
# Kategorien:
category_map = {
    "Klimawandel": "Neutral",
    "Klimakrise": "Alarmist",
}

# Neutrale und alarmistische Begriffe für diese Analyse behalten
df_neutral_alarm = df_drei[df_drei["begriff"].isin(category_map.keys())].copy()

df_neutral_alarm["category"] = df_neutral_alarm["begriff"].apply(
    lambda term: map_term_category(term, category_map)
 )

# Nach Quartal aggregieren
neutral_alarm_counts = df_neutral_alarm.groupby(["quarter", "category"]).size().unstack(fill_value=0)
neutral_alarm_total = neutral_alarm_counts.sum(axis=1)
neutral_alarm_pct = neutral_alarm_counts.div(neutral_alarm_total, axis=0) * 100

print("\n=== ANALYSIS 3: Neutral vs. alarmist per quarter (%) ===")
print(neutral_alarm_pct.round(2).to_string())


## 4. Abbildungen

Die folgenden Plots sind die zentralen Grafik-Exporte für die Studienarbeit.


### Grafik 1: Zeitliche Entwicklung der drei Begriffe


In [ ]:
# Monthly view directly from the already lemmatized/processed main frame
monthly_counts = (
    df_drei.assign(month=df_drei["data_published"].dt.to_period("M"))
    .groupby(["month", "begriff"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Klimaschutz", "Klimawandel", "Klimakrise"], fill_value=0)
)

monthly_counts["Gesamt"] = monthly_counts.sum(axis=1)

print(f"Period (min/max): {df_drei['data_published'].min()} to {df_drei['data_published'].max()}")
print(f"Mentions in df_drei: {len(df_drei)}")
print("\nMentions per term:")
print(df_drei["begriff"].value_counts().to_string())
print("\nMonthly table (absolute counts):")
print(monthly_counts.to_string())
print("\nMonthly table head:")
print(monthly_counts.head(10).to_string())
print("\nLast month:")
print(monthly_counts.tail(1).to_string())


In [ ]:
# Monatliche Werte robust gegen abgeschnittene Monate (pro beobachtetem Tag)
terms = ["Klimaschutz", "Klimawandel", "Klimakrise"]

df_terms = df_drei.copy()
df_terms["data_published"] = pd.to_datetime(df_terms["data_published"], errors="coerce")
df_terms = df_terms.dropna(subset=["data_published"]).copy()
df_terms["pub_day"] = df_terms["data_published"].dt.normalize()
df_terms["month"] = df_terms["pub_day"].dt.to_period("M")

df_days = df_meta_filtered.copy()
df_days["data_published"] = pd.to_datetime(df_days["data_published"], errors="coerce")
df_days = df_days.dropna(subset=["data_published"]).copy()
df_days["pub_day"] = df_days["data_published"].dt.normalize()
df_days["month"] = df_days["pub_day"].dt.to_period("M")

month_index = pd.period_range(df_days["month"].min(), df_days["month"].max(), freq="M")

monthly_counts = (
    df_terms.groupby(["month", "begriff"])
    .size()
    .unstack(fill_value=0)
    .reindex(month_index, fill_value=0)
    .reindex(columns=terms, fill_value=0)
)

observed_days = (
    df_days.groupby("month")["pub_day"]
    .nunique()
    .reindex(month_index, fill_value=0)
    .astype("float64")
)

monthly_values = monthly_counts.div(observed_days.replace(0, np.nan), axis=0).fillna(0)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(monthly_values))
labels = [str(m) for m in monthly_values.index]

ax.plot(x, monthly_values["Klimaschutz"], linestyle="-", marker="o", color="black", linewidth=1.8, markersize=4, label="Klimaschutz")
ax.plot(x, monthly_values["Klimawandel"], linestyle="--", marker="s", color="dimgray", linewidth=1.8, markersize=4, label="Klimawandel")
ax.plot(x, monthly_values["Klimakrise"], linestyle=":", marker="^", color="gray", linewidth=1.8, markersize=4, label="Klimakrise")

ax.set_xticks(x[::2])
ax.set_xticklabels(labels[::2], rotation=45, ha="right")
ax.set_xlabel("Monat")
ax.set_ylabel("Nennungen pro beobachtetem Tag")
ax.set_title("Monatliche Häufigkeiten der Klimabegriffe (pro beobachtetem Tag)")
ax.grid(True, alpha=0.3)
ax.legend(loc="best", frameon=True)

plt.tight_layout()
out_path = plot_dir / "06_monatliche_haeufigkeiten_pro_tag.png"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Plot gespeichert: {out_path}")

In [ ]:
# Daten vorbereiten
quarterly_pct_plot = quarterly_pct.drop(columns=["Gesamt"], errors="ignore")

fig, ax = plt.subplots(figsize=(10, 6))

# Linienarten für Graustufen-Darstellung
linestyles = {"Klimaschutz": "-", "Klimawandel": "--", "Klimakrise": ":"}
markers = {"Klimaschutz": "o", "Klimawandel": "s", "Klimakrise": "^"}

for begriff in ["Klimaschutz", "Klimawandel", "Klimakrise"]:
    if begriff in quarterly_pct_plot.columns:
        ax.plot(
            range(len(quarterly_pct_plot)),
            quarterly_pct_plot[begriff],
            linestyle=linestyles[begriff],
            marker=markers[begriff],
            color="black",
            linewidth=1.5,
            markersize=5,
            label=begriff,
        )

# X-Achsenbeschriftung
ax.set_xticks(range(len(quarterly_pct_plot)))
ax.set_xticklabels([str(q) for q in quarterly_pct_plot.index], rotation=45, ha="right")

ax.set_xlabel("Quartal")
ax.set_ylabel("Relativer Anteil (%)")
ax.set_title("Entwicklung der Klimabegriffe auf deutschsprachigen Titelseiten (2022–2025)")
ax.legend(loc="best", frameon=True)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(plot_dir, "06_grafik_1_zeitliche_entwicklung.png"), dpi=300, bbox_inches="tight")
plt.show()
print(f"Plot gespeichert: {os.path.join(plot_dir, '06_grafik_1_zeitliche_entwicklung.png')}")

### Grafik 2a: Neutral vs. alarmistisch als Flächendiagramm


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Flächendiagramm in Graustufen
x = range(len(neutral_alarm_pct))

ax.fill_between(
    x,
    0,
    neutral_alarm_pct["Neutral"],
    color="lightgray",
    label="Neutral (Klimawandel)",
    alpha=0.8,
)
ax.fill_between(
    x,
    neutral_alarm_pct["Neutral"],
    100,
    color="darkgray",
    label="Alarmistisch (Klimakrise)",
    alpha=0.8,
)

# X-Achsenbeschriftung
ax.set_xticks(x)
ax.set_xticklabels([str(q) for q in neutral_alarm_pct.index], rotation=45, ha="right")

ax.set_xlabel("Quartal")
ax.set_ylabel("Anteil (%)")
ax.set_title("Anteil neutraler und alarmistischer Begriffe im Zeitverlauf")
ax.legend(loc="upper right", frameon=True)
ax.set_ylim(0, 100)

plt.tight_layout()
plt.savefig(os.path.join(plot_dir, "06_grafik_2a_neutral_vs_alarmistisch_flaeche.png"), dpi=300, bbox_inches="tight")
plt.show()
print(f"Plot gespeichert: {os.path.join(plot_dir, '06_grafik_2a_neutral_vs_alarmistisch_flaeche.png')}")

### Grafik 2b: Neutral vs. alarmistisch als Liniendiagramm


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

x = range(len(neutral_alarm_pct))

ax.plot(
    x,
    neutral_alarm_pct["Neutral"],
    linestyle="-",
    marker="o",
    color="black",
    linewidth=2,
    label="Neutral (Klimawandel)",
)
ax.plot(
    x,
    neutral_alarm_pct["Alarmist"],
    linestyle="--",
    marker="s",
    color="gray",
    linewidth=2,
    label="Alarmistisch (Klimakrise)",
)

# X-Achsenbeschriftung
ax.set_xticks(x)
ax.set_xticklabels([str(q) for q in neutral_alarm_pct.index], rotation=45, ha="right")

ax.set_xlabel("Quartal")
ax.set_ylabel("Anteil (%)")
ax.set_title("Anteil neutraler und alarmistischer Begriffe im Zeitverlauf")
ax.legend(loc="best", frameon=True)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(plot_dir, "06_grafik_2b_neutral_vs_alarmistisch_linien.png"), dpi=300, bbox_inches="tight")
plt.show()
print(f"Plot gespeichert: {os.path.join(plot_dir, '06_grafik_2b_neutral_vs_alarmistisch_linien.png')}")

### Grafik 3: Absolute Häufigkeiten


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Data
begriffe = ["Klimaschutz", "Klimawandel", "Klimakrise"]
counts_sorted = [counts.get(b, 0) for b in begriffe]

# Balken in Graustufen
colors = ["black", "gray", "lightgray"]
bars = ax.bar(begriffe, counts_sorted, color=colors, edgecolor="black")

# Werte direkt an den Balken beschriften
for bar, count in zip(bars, counts_sorted):
    ax.annotate(
        f"{count:,}",
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 5),
        textcoords="offset points",
        ha="center",
        fontsize=11,
        fontweight="bold",
    )

ax.set_xlabel("Begriff")
ax.set_ylabel("Nennungen")
ax.set_title("Absolute Häufigkeiten der Klimabegriffe (2022–2025)")
ax.set_ylim(0, max(counts_sorted) * 1.15)

plt.tight_layout()
plt.savefig(os.path.join(plot_dir, "06_grafik_3_absolute_haeufigkeiten.png"), dpi=300, bbox_inches="tight")
plt.show()
print(f"Plot gespeichert: {os.path.join(plot_dir, '06_grafik_3_absolute_haeufigkeiten.png')}")

## EXKURS 5. Veränderungen nach Zeitungen (für weiterführende Untersuchungen interessant)


In [ ]:
TERMS = ['Klimaschutz', 'Klimawandel', 'Klimakrise']
terms_df = df_context.merge(
    df_meta_filtered[['newspaper_id', 'newspaper_name', 'data_published']],
    on='newspaper_id',
    how='inner',
)
terms_df['data_published'] = pd.to_datetime(terms_df['data_published'])
terms_df['begriff'] = (
    terms_df['suffix_lemma']
    .str.lower()
    .str.strip()
    .map(SUFFIX_TO_BEGRIFF)
)
terms_df = terms_df[terms_df['begriff'].notna()].copy()
terms_df['month'] = terms_df['data_published'].dt.to_period('M')
terms_df['quarter'] = terms_df['data_published'].dt.to_period('Q')


In [ ]:
df_q2q3 = delta_between_periods(terms_df, '2023Q2', '2023Q3', 'quarter')
scatter_delta(
    df_q2q3,
    'Klimakrise',
    'Klimaschutz',
    'Zeitungen: Veränderung 2023Q2 -> 2023Q3 (Klimakrise vs Klimaschutz)',
    label_threshold=20,
    plot_dir=plot_dir,
    outname='06_exkurs_5_delta_2023Q2_zu_2023Q3_krise_vs_schutz.png',
)


In [ ]:
df_delta_jun_jul = delta_between_periods(terms_df, '2023-06', '2023-07', 'month')
scatter_delta(
    df_delta_jun_jul,
    'Klimawandel',
    'Klimaschutz',
    'Zeitungen: Veränderung 2023-06 -> 2023-07 (Wandel vs Schutz)',
    label_threshold=20,
    plot_dir=plot_dir,
    outname='06_exkurs_5_delta_2023-06_zu_2023-07_wandel_vs_schutz.png',
)


In [ ]:
min_date = terms_df['data_published'].min()
max_date = terms_df['data_published'].max()
all_quarters = pd.period_range(min_date.to_period('Q'), max_date.to_period('Q'), freq='Q')
full_quarters = [q for q in all_quarters if q.start_time >= min_date and q.end_time <= max_date]
if len(full_quarters) < 2:
    print('Nicht genug vollständige Quartale gefunden.')
    first_q = None
    last_q = None
    df_delta_q = None
else:
    first_q = str(full_quarters[0])
    last_q = str(full_quarters[-1])
    print('Vergleichsquartale:', first_q, '->', last_q)
    df_delta_q = delta_between_periods(terms_df, first_q, last_q, 'quarter')


In [ ]:
if df_delta_q is not None:
    for term_x, term_y in combinations(TERMS, 2):
        scatter_delta(
            df_delta_q,
            term_x,
            term_y,
            f'Zeitungen: Veränderung {first_q} -> {last_q} ({term_x} vs {term_y})',
            label_threshold=40,
            plot_dir=plot_dir,
            outname=f'06_exkurs_5_delta_{first_q}_zu_{last_q}_{term_x}_vs_{term_y}.png',
        )


## 6. Befunde


In [ ]:
# Summary of key numbers
print("\n" + "=" * 60)
print("SUMMARY OF RESULTS")
print("=" * 60)

analysis_end_label = analysis_end.date() if analysis_end is not None else "open end"
print(f"\nTOTAL COUNTS ({analysis_start.date()} to {analysis_end_label})")
print(f"   Total mentions: {total:,}")
for b in begriffe:
    c = counts.get(b, 0)
    p = (c / total * 100)
    print(f"   {b}: {c:,} ({p:.1f}%)")

print("\nTIME DEVELOPMENT (first vs. last quarter)")
first_q = quarterly_pct_plot.iloc[0]
last_q = quarterly_pct_plot.iloc[-1]
for b in begriffe:
    if b in first_q.index:
        diff = last_q[b] - first_q[b]
        trend = "up" if diff > 0 else "down" if diff < 0 else "flat"
        print(f"   {b}: {first_q[b]:.1f}% -> {last_q[b]:.1f}% ({trend}, {diff:+.1f} pp)")
